# H2-1 확장 — 4단계: Type × 계층 셀 크기 확인

**목적**: 6단계 본 회귀에 들어가기 전, Type(A/B/C) × 계층(3계층/5분위) 조합별로 표본이
충분한지 확인. VIF 진단(3단계 후 별도 확인)은 통과했으므로, 이제 표본 크기만 확인하면
6단계로 넘어갈 준비가 끝난다.

**판단 기준**: 셀 크기 30명 미만이면 해당 조합의 회귀 결과는 참고용으로만 취급.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

DATA_DIR = "data/"
TIER_MIN_WEEK, TIER_MAX_WEEK = 17, 32
CAMP_MIN_WEEK, CAMP_MAX_WEEK = 33, 101


## (재구성) 1~3단계 — 이전과 동일한 파이프라인

In [2]:
tx = pd.read_csv(DATA_DIR + "transaction_data.csv", usecols=["household_key", "DAY", "WEEK_NO", "SALES_VALUE"])
campaign_table = pd.read_csv(DATA_DIR + "campaign_table.csv")
campaign_desc  = pd.read_csv(DATA_DIR + "campaign_desc.csv")

tier_window = tx[(tx["WEEK_NO"] >= TIER_MIN_WEEK) & (tx["WEEK_NO"] <= TIER_MAX_WEEK)]
n_weeks_tier = TIER_MAX_WEEK - TIER_MIN_WEEK + 1
avg_weekly_spend = (tier_window.groupby("household_key")["SALES_VALUE"].sum() / n_weeks_tier).rename("avg_weekly_spend_pre")
all_households = tx["household_key"].unique()
avg_weekly_spend = avg_weekly_spend.reindex(all_households).fillna(0)
tier5 = pd.qcut(avg_weekly_spend, 5, labels=["1분위(최저)", "2분위", "3분위", "4분위", "5분위(최고)"])
tier3 = pd.qcut(avg_weekly_spend, 3, labels=["저지출", "중지출", "고지출"])
tier_df = pd.DataFrame({"avg_weekly_spend_pre": avg_weekly_spend, "tier5": tier5, "tier3": tier3})

day_to_week = tx[["DAY", "WEEK_NO"]].drop_duplicates().sort_values("DAY").reset_index(drop=True)
day_arr, week_arr = day_to_week["DAY"].values, day_to_week["WEEK_NO"].values
def day_to_week_lookup(day):
    idx = np.searchsorted(day_arr, day, side="right") - 1
    return week_arr[max(0, min(idx, len(day_arr) - 1))]
campaign_desc = campaign_desc.copy()
campaign_desc["START_WEEK"] = campaign_desc["START_DAY"].apply(day_to_week_lookup)
campaign_desc["END_WEEK"] = campaign_desc["END_DAY"].apply(day_to_week_lookup)
camp_full = campaign_table.merge(
    campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_WEEK", "END_WEEK"]],
    on="CAMPAIGN", how="left", suffixes=("", "_desc")
)
recipients = set(campaign_table["household_key"].unique())
never_recipients = set(all_households) - recipients

weeks_camp = list(range(CAMP_MIN_WEEK, CAMP_MAX_WEEK + 1))
panel_index = pd.MultiIndex.from_product([all_households, weeks_camp], names=["household_key", "WEEK_NO"])
panel = pd.DataFrame(index=panel_index).reset_index()
weekly_spend = (
    tx[(tx["WEEK_NO"] >= CAMP_MIN_WEEK) & (tx["WEEK_NO"] <= CAMP_MAX_WEEK)]
    .groupby(["household_key", "WEEK_NO"])["SALES_VALUE"].sum().rename("spend")
)
panel = panel.merge(weekly_spend, on=["household_key", "WEEK_NO"], how="left")
panel["spend"] = panel["spend"].fillna(0)

def build_active_weeks(camp_df, type_name):
    sub = camp_df[camp_df["DESCRIPTION_desc"] == type_name][["household_key", "START_WEEK", "END_WEEK"]].copy()
    sub["START_WEEK"] = sub["START_WEEK"].clip(lower=CAMP_MIN_WEEK)
    sub["END_WEEK"] = sub["END_WEEK"].clip(upper=CAMP_MAX_WEEK)
    sub["WEEK_NO"] = sub.apply(lambda r: list(range(int(r["START_WEEK"]), int(r["END_WEEK"]) + 1)), axis=1)
    sub = sub.explode("WEEK_NO")[["household_key", "WEEK_NO"]].drop_duplicates()
    sub["WEEK_NO"] = sub["WEEK_NO"].astype(int)
    sub[f"active_{type_name}"] = 1
    return sub

for t in ["TypeA", "TypeB", "TypeC"]:
    active = build_active_weeks(camp_full, t)
    panel = panel.merge(active, on=["household_key", "WEEK_NO"], how="left")
    panel[f"active_{t}"] = panel[f"active_{t}"].fillna(0).astype(int)

panel = panel.merge(tier_df[["tier3", "tier5"]], left_on="household_key", right_index=True, how="left")
print("파이프라인 재구성 완료. panel shape:", panel.shape)


파이프라인 재구성 완료. panel shape: (172500, 8)


## 4단계 — Type × 계층 셀 크기표

각 Type(A/B/C)마다, 33~101주 동안 **한 번이라도** 해당 타입 캠페인을 받은 고유 가구 수를 계층별로 집계.

In [3]:
def cell_size_table(tier_col):
    out = {}
    for t in ["TypeA", "TypeB", "TypeC"]:
        hh_active = panel.loc[panel[f"active_{t}"] == 1, ["household_key", tier_col]].drop_duplicates()
        out[t] = hh_active.groupby(tier_col, observed=True)["household_key"].nunique()
    return pd.DataFrame(out)

print("[Type x 3계층 - 수신 가구 수 (본 분석용)]")
cell3 = cell_size_table("tier3")
print(cell3)

print()
print("[Type x 5분위 - 수신 가구 수 (보조 분석용)]")
cell5 = cell_size_table("tier5")
print(cell5)


[Type x 3계층 - 수신 가구 수 (본 분석용)]
       TypeA  TypeB  TypeC
tier3                     
저지출      187     90     28
중지출      554    305     74
고지출      772    628    295

[Type x 5분위 - 수신 가구 수 (보조 분석용)]
         TypeA  TypeB  TypeC
tier5                       
1분위(최저)     74     42     16
2분위        192     92     25
3분위        328    178     40
4분위        444    293     92
5분위(최고)    475    418    224


In [4]:
THRESH = 30
print(f"판단 기준: 셀 크기 {THRESH}명 미만 -> 참고용으로만 취급")
print()

long3 = cell3.stack().reset_index()
long3.columns = ["tier3", "Type", "인원수"]
below3 = long3[long3["인원수"] < THRESH]
print("[3계층 기준 - 미달 셀]")
print(below3.to_string(index=False) if len(below3) > 0 else "없음 (모든 조합 30명 이상)")

print()
long5 = cell5.stack().reset_index()
long5.columns = ["tier5", "Type", "인원수"]
below5 = long5[long5["인원수"] < THRESH]
print("[5분위 기준 - 미달 셀]")
print(below5.to_string(index=False) if len(below5) > 0 else "없음")


판단 기준: 셀 크기 30명 미만 -> 참고용으로만 취급

[3계층 기준 - 미달 셀]
tier3  Type  인원수
  저지출 TypeC   28

[5분위 기준 - 미달 셀]
  tier5  Type  인원수
1분위(최저) TypeC   16
    2분위 TypeC   25


**결과 요약**

- **3계층 기준**: 저지출×TypeC = **28명**만 기준(30명) 미달, 그것도 딱 2명 차이로 경계선. 나머지 8개 조합은 전부 74명 이상으로 안정적.
- **5분위 기준**: 1분위×TypeC = 16명, 2분위×TypeC = 25명, 두 조합이 기준 미달. 이건 이전에 팀원이 걱정했던
  "TypeC × 저지출 계층 표본 부족" 우려가 **5분위 단위에서는 실제로 확인된다**는 뜻.

→ 이게 바로 처음에 "3계층을 본 분석, 5분위를 보조 분석"으로 설계한 이유가 맞았다는 걸 데이터로 확인한 셈.
3계층 기준으로는 사실상 게이트 통과(저지출×TypeC 하나만 경계선), 5분위로 세분화하면 TypeC 쪽이 못 버팀.

### 보조 확인 — 계층별 캠페인 도달률(Reach)

팀원이 예시로 든 "1분위 22.1% 수신 vs 5분위 96.1% 수신"이 실제 데이터에서도 나타나는지 확인.

In [5]:
tier_df["received_any"] = tier_df.index.isin(recipients)

reach3 = tier_df.groupby("tier3", observed=True)["received_any"].agg(["sum", "count"])
reach3["도달률(%)"] = (reach3["sum"] / reach3["count"] * 100).round(1)
reach3.columns = ["수신가구수", "전체가구수", "도달률(%)"]
print("[3계층별 캠페인(전체 타입 통합) 도달률]")
print(reach3)

print()
reach5 = tier_df.groupby("tier5", observed=True)["received_any"].agg(["sum", "count"])
reach5["도달률(%)"] = (reach5["sum"] / reach5["count"] * 100).round(1)
reach5.columns = ["수신가구수", "전체가구수", "도달률(%)"]
print("[5분위별 캠페인(전체 타입 통합) 도달률]")
print(reach5)


[3계층별 캠페인(전체 타입 통합) 도달률]
       수신가구수  전체가구수  도달률(%)
tier3                      
저지출      210    834    25.2
중지출      591    834    70.9
고지출      783    832    94.1

[5분위별 캠페인(전체 타입 통합) 도달률]
         수신가구수  전체가구수  도달률(%)
tier5                        
1분위(최저)     89    500    17.8
2분위        208    500    41.6
3분위        353    500    70.6
4분위        453    500    90.6
5분위(최고)    481    500    96.2


In [6]:
reach_by_type = pd.DataFrame({
    t: cell3[t] / tier_df.groupby("tier3", observed=True).size() * 100
    for t in ["TypeA", "TypeB", "TypeC"]
}).round(1)
print("[3계층 x Type별 도달률(%)]")
print(reach_by_type)


[3계층 x Type별 도달률(%)]
       TypeA  TypeB  TypeC
tier3                     
저지출     22.4   10.8    3.4
중지출     66.4   36.6    8.9
고지출     92.8   75.5   35.5


**결과 요약**

- 3계층 통합 도달률: 저지출 **25.2%** vs 고지출 **94.1%** — 팀원이 예로 든 수치(22.1% vs 96.1%)와 거의 일치.
  **캠페인이 무작위로 뿌려진 게 아니라 원래 고지출 고객에게 집중됐다는 게 실제 데이터로 확인됨.**
  → "수신자 vs 비수신자" 단순비교를 하면 안 된다는 팀원의 우려가 실제로 맞았다는 근거.
- Type별로 보면 **TypeC의 상대적 쏠림이 가장 심함** (저지출 3.4% vs 고지출 35.5%, 약 10.4배 차이).
  TypeA(4.1배)·TypeB(7배)보다 훨씬 큼. TypeC가 표본도 가장 작은데 타겟팅 쏠림까지 가장 심하다는 뜻이라,
  결과 해석 시 TypeC 관련 결론은 특히 더 신중하게 다뤄야 함.

## 판정 — 4단계 게이트 통과 여부

| 항목 | 결과 | 판정 |
|---|---|---|
| 3계층 셀 크기 | 8/9 조합 74명 이상, 1개(저지출×TypeC) 28명 | **통과** (경계선 1개, 결론 시 각주 처리) |
| 5분위 셀 크기 | TypeC 쪽 2개 조합 미달 | 5분위는 보조 분석으로만 사용 (원래 계획대로) |
| 선택편향 확인 | 저/고지출 도달률 25.2% vs 94.1%로 매우 큼 | 6단계에서 가구 고정효과로 반드시 통제해야 함이 재확인됨 |

**다음 단계**: 5단계(평행추세 사전검정)로 진행 가능. 단, 저지출×TypeC 조합과 TypeC 전반은
6단계 결과에서 "경계선 표본"이라는 단서를 달고 해석할 것.

In [7]:
# 저지출×TypeC 수신 가구 (33~101주 중 한 번이라도 TypeC 활성)
low_typeC_hh = panel.loc[
    (panel["active_TypeC"] == 1) & (panel["tier3"] == "저지출"), "household_key"
].unique()
print(f"저지출×TypeC 수신 가구 수: {len(low_typeC_hh)}")

# 사전기간(17~32주) 거래 0이었던 가구와의 중첩 확인
tier_df["was_zero_pre"] = (tier_df["avg_weekly_spend_pre"] == 0)
overlap = tier_df.loc[tier_df.index.isin(low_typeC_hh), "was_zero_pre"]
n_overlap = overlap.sum()
print(f"그중 사전지출 0이었던 가구: {n_overlap} ({n_overlap/len(low_typeC_hh)*100:.1f}%)")

저지출×TypeC 수신 가구 수: 28
그중 사전지출 0이었던 가구: 2 (7.1%)
